# mjo_wavenum_freq_season

- "Calculates wavenumber-frequency spectra via seasonal averaging as defined by the US-CLIVAR MJO diagnostics website"
- [NCL Reference](https://www.ncl.ucar.edu/Document/Functions/Diagnostics/mjo_wavenum_freq_season.shtml)

### Example NCL Script and Output

- mjo_wavenum_freq_season.ncl
- mjo_output/mjo_wavenum_freq_season_output.txt

In [ ]:
import os
import numpy as np
import xarray as xr
from datetime import datetime

In [ ]:
wavenum_freq = np.loadtxt("mjo_output/mjo_wavenum_freq_season_winter_ncl_output.txt", skiprows=18)
ncl_wavenum_freq = wavenum_freq.reshape(289, 181)
print(f"NCL Wavenumbers: {len(ncl_wavenum_freq)}")
print(f"NCL Frequencies: {len(ncl_wavenum_freq[0])}")

MJO CLIVAR: Wave number-frequency spectra
- "winter": 180 days (starts November 1)
- "summer": 180 days (starts May 1)
- "summer": 365 days (starts January 1)

In [ ]:
u850_data = xr.open_dataset(os.getcwd() + "/data/anomaly/QBOi.EXP1.AMIP.001.u850.day.anom.nc")
u850_data

In [ ]:
# Filter out latitude and time ranges (based on NCL script)

## filter out latitude ranges
latS = -10
latN = 10

u850_data = u850_data.sel(lat=slice(latS, latN)) 

## filter out time ranges 
twStrt = "1979-01-01" # time window start
twLast = "1981-12-31" # time window end

u850_data = u850_data.sel(time=slice(twStrt, twLast)) 
u850_data

In [ ]:
# Average data over latitude and use the averaged to compute spectra
## Compute the average of latitude

def dim_avg_n_wrap_python(data, dim):
    # https://www.ncl.ucar.edu/Document/Functions/Contributed/dim_avg_n_Wrap.shtml
    data = data.mean(dim=dim)
    return data

u850_data = dim_avg_n_wrap_python(u850_data, "lat")
u850_data

MJO wavenumber-frequency spectra based on Level 2 diagnostics- Based on [US-CLIVAR MJO Working Group (2009) "MJO Simulation Diagnostics - Level 2 diagnostics"](https://doi.org/10.1175/2008JCLI2731.1)

> "Level 2 diagnostics are designed to explore more detailed features of the MJO. They include wavenumber-frequency spectra of individual fields, cross-spectral quantities between different fields, and a multivariate EOF analysis. Wavenumber-frequency spectra for equatorial precipitation and 850-hPa zonal wind are shown in Fig. 7 for boreal summer, and in Fig. 8 for boreal winter.

> The spectra were computed by Fourier transforming 180-day segments centered on boreal summer and boreal winter, forming power, and then averaging over all years of data (1979–2005). The resulting bandwidth is (180 days)−1. Only the climatological season cycle was removed before calculation of the spectra.

> By definition, eastward propagation is represented by positive frequency and positive wavenumber whereas westward propagation is represented with one or the other of the frequency or wavenumber being negative. If standing oscillations are present, they will project as equal amounts of power in eastward and westward directions. The results indicate a concentration of power at 30–90-day periods and zonal wavenumber 1 for 850-hPa zonal wind, and zonal wavenumbers 1–3 for precipitation and OLR (e.g., Salby and Hendon 1994)"

In [ ]:
seasonName = "winter"

In [ ]:
def mjo_wavenum_freq_season_as_python(input_data, data_var, seasonName):
    # Python equivalent of mjo_wavenum_freq_season()
    # Based on ncl: mjo_wavenum_freq_season
    # https://github.com/NCAR/ncl/blob/8f9e9476281cc6f6d9d12eaa78729c7003ca24b7/ni/src/examples/gsun/diagnostics_cam.ncl#L2886
    data = input_data[data_var]

    seasonName = seasonName.lower()
    print(f"Date for {seasonName}")
    if seasonName == "winter":
        # filter out boreal winter across multiple years
        # Winter: November, December, January, February, March, April
        data = data.sel(time=data.time.dt.month.isin([11, 12, 1, 2, 3, 4]))
    if seasonName == "summer":
        # filter out boreal summer across multiple years
        # Summer: May, June, July, August, September, October
        data = data.sel(time=data.time.dt.month.isin([5, 6, 7, 8, 9, 10]))

    time_step_dt = (data.time[1] - data.time[0]).astype('timedelta64[D]').item().days # time steps (in days)
    lon_step_dx = (data.lon[1] - data.lon[0]).item() # longitude (degree step)
    print(f"Time step: {time_step_dt} day(s) with lon step: {lon_step_dx} degrees")
        
    # Detrend data (over time)
    poly_coeffs = data.polyfit(dim="time", deg=lon_step_dx, skipna=True)
    fit = xr.polyval(data["time"], poly_coeffs["polyfit_coefficients"])
    data_detrend = data - fit
    data_detrend.attrs = data.attrs
    data_detrend.name = data.name

    # remove seasonal cycle from data before apply 2D FFT
    ## calculate mean seasonal climate for each season across time range
    seasonal_climatology_mean = data_detrend.groupby("time.season").mean("time")
    ## remove seasonal cycle (deseasonalize) from data
    data_deseason = data_detrend.groupby("time.season") - seasonal_climatology_mean

    n_time = len(data_deseason.time)
    m_lon = len(data_deseason.lon)

    # average Wavenumber-Frequency power spectra over seasons via 2D FFT
    ## 2D FFT on input data for specific season
    spectrum = np.fft.fft2(data_deseason.values)
    #spectrum = np.fft.fftn(data_deseason)
    power_spectral_density = np.abs(spectrum)**2 / (n_time * m_lon) ## Power Spectrum is the square magntidue
    
    ## Determine frequency
    freqs = np.fft.fftfreq(n_time, d=time_step_dt)
    
    ## Determine wavenumbers
    wavenumbers = np.fft.fftfreq(m_lon, d=lon_step_dx)
    
    ## Shift the zero frequency to the center
    wavenumbers = np.fft.fftshift(wavenumbers)
    freqs = np.fft.fftshift(freqs)
    power_spectral_density = np.fft.fftshift(power_spectral_density)
    
    ## Return data as Wavenumber X Frequency
    wavenum_freq = xr.DataArray(power_spectral_density,
                                 coords={"frequency": freqs, 
                                         "wavenumbers": wavenumbers},
                                 dims=["frequency", "wavenumbers"],
                                 name="wavenum_freq")

    print(f"[wavenumber | {len(wavenum_freq.wavenumbers)}] x [freq | {len(wavenum_freq.frequency)}]")
    return wavenum_freq

In [ ]:
wavenum_freq = mjo_wavenum_freq_season_as_python(u850_data, "U850", seasonName)
wavenum_freq

In [ ]:
# save output
ds = wavenum_freq.to_dataframe()
ds.to_csv(f"mjo_output/mjo_wavenum_freq_season_{seasonName}_python_output.txt", index=True)

# mjo_wavenum_freq_season_plot

- "Plot wavenumber-frequency spectra as returned by mjo_wavenum_freq_season"
- [NCL Reference](https://www.ncl.ucar.edu/Document/Functions/Diagnostics/mjo_wavenum_freq_season_plot.shtml)

### Example Plot fom `mjo_wavenum_freq_season.ncl`

- mjo_output/mjo_wavenum_freq_season_plot.winter.png

<center>
<img src="mjo_output/mjo_wavenum_freq_season_plot.winter.png" width="400" height="600">
</center>

In [ ]:
wavenum_freq = np.loadtxt(f"mjo_output/mjo_wavenum_freq_season_{seasonName}_ncl_output.txt", skiprows=18)
wavenum_freq

In [ ]:
# Load NCL data to generate a comparison plot
wavenum_freq = np.loadtxt(f"mjo_output/mjo_wavenum_freq_season_{seasonName}_ncl_output.txt", skiprows=18)
ncl_wavenum_freq = wavenum_freq.reshape(289, 181)
ncl_wavenum_freq_xr = xr.DataArray(ncl_wavenum_freq,
                                   coords={"frequency": 
                                   dims=["frequency", "wavenumbers"],
                                   name="wavenum_freq")
ncl_wavenum_freq_xr.sel(frequency=slice(-0.5, 0.5)) 
#ncl_wavenum_freq_xr

In [ ]:
import matplotlib.pyplot as plt

def mjo_wavenum_freq_season_plot_as_python(plt_title, wavenum_freq,
                                           max_wavenum=6,
                                           min_freq=-0.5, max_freq=0.5):
    fig, ax = plt.subplots(figsize=(6, 6))
    
    # filter wave numbers from 0 to 6 (default)
    wavenum_freq = wavenum_freq.sel(wavenumbers=slice(0, max_wavenum)) 
    # filter frequencies from -0.5 to 0.5 (default)
    wavenum_freq = wavenum_freq.sel(frequency=slice(min_freq, max_freq)) 

    # Plot
    plt.contour(wavenum_freq,
                vmax=(wavenum_freq).max(), vmin=(wavenum_freq).min(),
                levels=10)
    plt.imshow(wavenum_freq, 
               vmax=(wavenum_freq).max(), vmin=(wavenum_freq).min(), 
               aspect="auto")
    
    #ax.set_xlim([x_min, x_max])
    #ax.set_ylim([y_min, y_max])
    plt.title(f"Wavenumber-Frequency: {plt_title}")
    plt.xlabel("Frequency (cycles/day)")
    plt.ylabel("Zonal Wavenumber (cycles/degree)")
    #plt.grid(True)
    plt.show()

In [ ]:
mjo_wavenum_freq_season_plot_as_python(f"{seasonName.upper()}", ncl_wavenum_freq_xr)

In [ ]:
#mjo_wavenum_freq_season_plot_as_python(f"NCL -> {seasonName.upper()}", ncl_wavenum_freq)

In [ ]:
## Old Version:
import matplotlib.pyplot as plt

def mjo_wavenum_freq_season_plot_as_python(seasonName, power_spectrum):
    fig, ax = plt.subplots(figsize=(6, 6))
    power_spectrum.plot.pcolormesh()
    plt.title(f"Wavenumber-Frequency: {seasonName.upper()}")
    plt.xlabel("Frequency (cycles/day)")
    plt.ylabel("Zonal Wavenumber (cycles/degree)")
    #plt.grid(True)
    plt.show()